In [3]:
import firebase_admin
from firebase_admin import credentials, db
import pandas as pd
import datetime
import requests 
import pytz
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tabulate import tabulate
import time

# --------------- Firebase Initialization ---------------
if not firebase_admin._apps:
    cred = credentials.Certificate("E:\\flood-web\\serviceAccountKey.json")  # Adjust path if needed
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://my-water-flow-project-f1f53-default-rtdb.firebaseio.com/'
    })

firebase_ref = db.reference('/FloodPrediction')

# --------------- OpenWeatherMap Setup ------------------
API_KEY = "616986177c8131321d11803ed44df570"
cities = ["Papanasam", "Aryankavu", "Thenmala", "Kalakkad", "Thenkasi"]

# --------------- Load Dataset and Train Model ----------
df = pd.read_csv('waterfall_data.csv')

features = ["Papanasam", "Aryankavu", "Thenmala", "Kalakkad", "Thenkasi", "Total Rainfall (mm)"]
target = "Flood Risk"

le = LabelEncoder()
df[target] = le.fit_transform(df[target])

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --------------- Rainfall Data Function ----------------
def get_weather_data(city):
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&units=metric&cnt=2&appid={API_KEY}"
    response = requests.get(url)
    data = response.json()

    if response.status_code != 200 or 'list' not in data:
        print(f"Error fetching data for {city}. Response: {data}")
        return 0

    rainfall = sum([forecast.get('rain', {}).get('3h', 0) for forecast in data['list']])
    return rainfall

def get_all_rainfall_data():
    rainfall_data = {}
    for city in cities:
        rainfall_data[city] = get_weather_data(city)
    return rainfall_data

# --------------- Prediction Function -------------------
def predict_flood_risk(rainfall_data):
    X_new = pd.DataFrame([rainfall_data])
    prediction = model.predict(X_new)
    prediction_label = le.inverse_transform(prediction)[0]
    return prediction_label

# --------------- Dashboard Display ---------------------
def display_dashboard(rainfall_data):
    total_rainfall = rainfall_data["Total Rainfall (mm)"]
    table_data = [[city, f"{rain:.2f} mm"] for city, rain in rainfall_data.items()]
    headers = ["City", "Rainfall Rate"]
    print("\n🌧 Rainfall Rates from 5 Cities:\n")
    print(tabulate(table_data, headers=headers, tablefmt="pretty"))
    print(f"\n💧 Total Rainfall: {total_rainfall:.2f} mm")

# --------------- Main Update Loop ----------------------
def update_firebase():
    while True:
        rainfall_data = get_all_rainfall_data()
        total_rainfall = sum(rainfall_data.values())
        rainfall_data["Total Rainfall (mm)"] = total_rainfall

        prediction_label = predict_flood_risk(rainfall_data)

        note = "Potential Risk and Caution Tourists at Coutrallam, due to present weather condition"
        action = "Issue Flood Alert and Caution Tourists" if prediction_label == "Yes" else \
                 "Monitor Closely – Potential Risk and Caution Tourists" if prediction_label == "Unpredictable" else \
                 "No Immediate Action Required"

        india_time = datetime.datetime.now(pytz.timezone("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S")

        firebase_data = {
            "FallsName": "Coutralam Falls",
            "PredictedDate": (datetime.datetime.utcnow() + datetime.timedelta(days=1)).strftime("%Y-%m-%d"),
            "PredictedFloodRisk": prediction_label,
            "RecommendedAction": action,
            "Note": note,
            "Time": india_time
        }

        firebase_ref.set(firebase_data)

        print("\n Updated Data Sent to Firebase:")
        print(tabulate([firebase_data], headers="keys", tablefmt="pretty"))

        # Display Rainfall Dashboard in Notebook
        display_dashboard(rainfall_data)

        # ⏱ Wait 60 seconds before next update
        time.sleep(60)

# --------------- Start the Prediction Process ----------
update_firebase()


 Updated Data Sent to Firebase:
+-----------------+---------------+--------------------+------------------------------+-------------------------------------------------------------------------------------+---------------------+
|    FallsName    | PredictedDate | PredictedFloodRisk |      RecommendedAction       |                                        Note                                         |        Time         |
+-----------------+---------------+--------------------+------------------------------+-------------------------------------------------------------------------------------+---------------------+
| Coutralam Falls |  2025-04-08   |         No         | No Immediate Action Required | Potential Risk and Caution Tourists at Coutrallam, due to present weather condition | 2025-04-07 14:13:02 |
+-----------------+---------------+--------------------+------------------------------+-------------------------------------------------------------------------------------+----------

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))